In [ ]:
# @title 🛠️ 1. Cài đặt Thư viện Studio & Kết nối Google Drive
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
!pip install -q curl_cffi google-genai edge-tts openai-whisper ipywidgets pillow requests moviepy ffmpeg-python nest_asyncio
import nest_asyncio
nest_asyncio.apply()
print("✅ Hoàn tất cài đặt môi trường Anime Studio!")

In [ ]:
# @title ⚙️ 2. Core Engine (AI Phân Cảnh 30 Ảnh English + Phụ Đề Vàng Giữa Khung + Full MP4 Render)
import warnings
warnings.filterwarnings('ignore')
import os, sys, time, json, hashlib, re, urllib.parse, asyncio, random, shutil, subprocess
from pathlib import Path
import requests
from PIL import Image, ImageDraw, ImageFont
from curl_cffi import requests as cffi_requests
import nest_asyncio
nest_asyncio.apply()

try:
    from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips
except Exception:
    from moviepy.video.io.ImageSequenceClip import ImageSequenceClip

TARGET_W, TARGET_H = 1080, 1920
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
BASE_LIBRARY_DIR = Path('/content/drive/MyDrive/anime_library')

# -----------------------------------------------------
# 1. THUẬT TOÁN CÀO TRỰC TIẾP PINTEREST WEB PINS
# -----------------------------------------------------
def search_pinterest_direct(query, limit=50):
    query_clean = urllib.parse.quote(query)
    search_url = f"https://www.pinterest.com/search/pins/?q={query_clean}"
    session = cffi_requests.Session()
    urls = []
    bookmarks = []
    try:
        r1 = session.get(search_url, impersonate="chrome124")
        csrf_token = session.cookies.get("csrftoken") or "123456"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "application/json, text/javascript, */*, q=0.01",
            "X-Requested-With": "XMLHttpRequest",
            "X-CSRFToken": csrf_token,
            "X-Pinterest-AppState": "active",
            "X-Pinterest-PWS-Handler": "www/search/pins.js",
            "Referer": search_url,
        }
        for page in range(5):
            options = {"isPrefetch": False, "query": query, "scope": "pins", "no_fetch_context_on_resource": False}
            if bookmarks: options["bookmarks"] = bookmarks
            params = {"source_url": f"/search/pins/?q={query_clean}", "data": json.dumps({"options": options, "context": {}}), "_": str(int(time.time() * 1000))}
            api_url = "https://www.pinterest.com/resource/BaseSearchResource/get/"
            r2 = session.get(api_url, params=params, headers=headers, impersonate="chrome124")
            if r2.status_code == 200:
                res_resp = r2.json().get("resource_response", {})
                results = res_resp.get("data", {}).get("results", [])
                new_b = res_resp.get("bookmark")
                if new_b: bookmarks = [new_b]
                for pin in results:
                    images = pin.get("images", {})
                    orig = images.get("orig", {}).get("url") or images.get("736x", {}).get("url") or images.get("474x", {}).get("url")
                    if orig and orig not in urls: urls.append(orig)
                if len(urls) >= limit or not new_b: break
            else: break
            time.sleep(1)
    except Exception as e:
        print(f"Lỗi kết nối Pinterest Web: {e}")
    seen = set()
    unique = [u for u in urls if not (u in seen or seen.add(u))]
    return unique[:limit]

def resize_crop_save(media_data, out_path):
    tmp = out_path.parent / f"_tmp_{out_path.name}"
    tmp.write_bytes(media_data)
    try:
        img = Image.open(tmp).convert('RGB')
        w, h = img.size
        ratio = TARGET_W / TARGET_H
        if w/h > ratio: nh, nw = TARGET_H, int(w * (TARGET_H / h))
        else: nw, nh = TARGET_W, int(h * (TARGET_W / w))
        img = img.resize((nw, nh), Image.LANCZOS)
        l, t = (nw - TARGET_W) // 2, (nh - TARGET_H) // 2
        img.crop((l, t, l + TARGET_W, t + TARGET_H)).save(out_path, 'JPEG', quality=92)
        tmp.unlink(missing_ok=True)
        return True
    except Exception:
        tmp.unlink(missing_ok=True)
        return False

def build_library(char_key, anime_name, base_dir, target=50):
    char_dir = base_dir / char_key
    char_dir.mkdir(parents=True, exist_ok=True)
    existing = list(char_dir.glob("*.jpg")) + list(char_dir.glob("*.png")) + list(char_dir.glob("*.jpeg")) + list(char_dir.glob("*.webp"))
    if len(existing) >= target:
        print(f"  ✅ [{char_key}]: Đã đủ {len(existing)}/{target} ảnh yêu cầu! (Bỏ qua không tải nữa)")
        return
    used_hashes = {hashlib.md5(f.read_bytes()).hexdigest() for f in existing if f.exists()}
    query = f"{char_key.replace('_', ' ')} {anime_name.replace('_', ' ')}"
    print(f"🔎 Đang cào Pinterest Pins cho '{query}' (Hiện có: {len(existing)}/{target})...")
    urls = search_pinterest_direct(query, limit=target * 2)
    saved_count = len(existing)
    for url in urls:
        if saved_count >= target: break
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code != 200 or len(r.content) < 8000: continue
            h = hashlib.md5(r.content).hexdigest()
            if h in used_hashes: continue
            used_hashes.add(h)
            out_file = char_dir / f"{char_key}_{saved_count+1:02d}.jpg"
            if resize_crop_save(r.content, out_file):
                saved_count += 1
                print(f"    + [{char_key}] #{saved_count:02d}: Đã lưu Pinterest Pin!")
        except Exception: continue
    print(f"  🎉 HOÀN THÀNH [{char_key}]: {saved_count}/{target} ảnh Pinterest!")

def run_fetch(anime_name, char_list=None, single_char=None, target_per_char=50):
    anime_dir = BASE_LIBRARY_DIR / anime_name
    conf_path = anime_dir / "characters_config.json"
    if not conf_path.exists():
        print(f"LỖI: Chưa có file characters_config.json cho '{anime_name}'!")
        return
    try: char_dict = json.loads(conf_path.read_text(encoding="utf-8"))
    except Exception as e: print(f"LỖI đọc config: {e}"); return
    
    if single_char:
        target_chars = [single_char]
    elif char_list and len(char_list) > 0:
        target_chars = list(char_list)
    else:
        target_chars = list(char_dict.keys())
    
    print(f"\n{'='*50}\n🚀 TẢI PINTEREST CHO {len(target_chars)} NV TRONG: {anime_name} (Chỉ tiêu: {target_per_char} ảnh/NV)\n{'='*50}")
    for char_key in target_chars:
        build_library(char_key, anime_name, anime_dir, target=target_per_char)

# -----------------------------------------------------
# 2. HỆ THỐNG AI TẠO KỊCH BẢN BẮT BUỘC TIẾNG ANH & PHÂN CẢNH 30 ẢNH
# -----------------------------------------------------
def generate_script_gemini(topic, anime_name, available_chars, api_key):
    models = ["gemini-3.1-flash-lite", "gemini-2.0-flash", "gemini-1.5-flash", "gemini-2.5-flash"]
    chars_str = ", ".join(available_chars) if available_chars else anime_name
    prompt = f"""You are an expert anime Short video director. Write a viral 60-second narrative script in ENGLISH about '{topic}' for anime '{anime_name}'.
CRITICAL MANDATE: Regardless of what language the topic is provided in, the generated script and voiceover text MUST BE WRITTEN ENTIRELY IN ENGLISH!

Available character keys in this anime library: [{chars_str}]

REQUIREMENTS:
1. The full script must be 150-160 English words (~60 seconds reading duration).
2. Divide the script into EXACTLY 30 scenes (each scene corresponds to ~2 seconds of English narration).
3. For EACH scene, assign the most relevant 'character_key' from the available list: [{chars_str}]. If a scene refers to general events, assign the main protagonist character key.

Return STRICTLY valid JSON with structure:
{{
  "script": "Full narrative script text in English...",
  "tts_script": "Full voiceover text in English...",
  "scenes": [
    {{
      "scene_index": 1,
      "text_snippet": "Short English text spoken in this 2s scene",
      "character_key": "Character_Name_Key"
    }}
  ]
}}"""
    body = {'contents': [{'parts': [{'text': prompt}]}], 'generationConfig': {'responseMimeType': 'application/json'}}
    for model in models:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={api_key}"
        try:
            r = requests.post(url, json=body, timeout=60)
            if r.status_code == 200:
                text = r.json()['candidates'][0]['content']['parts'][0]['text']
                return json.loads(text)
            else:
                print(f"⚠️ Thử model {model} (Status {r.status_code})...")
        except Exception as e:
            print(f"⚠️ Lỗi model {model}: {e}")
    return None

def pick_unique_scene_images(scenes, anime_name):
    anime_dir = BASE_LIBRARY_DIR / anime_name
    char_images_map = {}
    if anime_dir.exists():
        for cdir in anime_dir.iterdir():
            if cdir.is_dir() and cdir.name != "output_shorts":
                imgs = list(cdir.glob("*.jpg")) + list(cdir.glob("*.png")) + list(cdir.glob("*.jpeg")) + list(cdir.glob("*.webp"))
                random.shuffle(imgs)
                char_images_map[cdir.name] = imgs
            
    used_images = set()
    all_anime_imgs = [img for imgs in char_images_map.values() for img in imgs]
    random.shuffle(all_anime_imgs)
    
    selected_timeline = []
    for sc in scenes:
        s_idx = sc.get('scene_index', len(selected_timeline)+1)
        ckey = sc.get('character_key', '')
        snippet = sc.get('text_snippet', '')
        chosen_img = None
        
        if ckey in char_images_map:
            for img in char_images_map[ckey]:
                if str(img) not in used_images:
                    chosen_img = img
                    break
        
        if not chosen_img:
            for img in all_anime_imgs:
                if str(img) not in used_images:
                    chosen_img = img
                    break
                    
        if not chosen_img and all_anime_imgs:
            chosen_img = random.choice(all_anime_imgs)
            
        if chosen_img:
            used_images.add(str(chosen_img))
            selected_timeline.append({
                "scene": s_idx,
                "character": ckey,
                "text": snippet,
                "image": chosen_img.name,
                "image_path": str(chosen_img)
            })
    return selected_timeline

async def _edge_tts_save(text, voice, out_mp3):
    import edge_tts
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(str(out_mp3))

def generate_tts_robust(text, voice, out_mp3):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            loop.create_task(_edge_tts_save(text, voice, out_mp3))
            for _ in range(30):
                if out_mp3.exists() and out_mp3.stat().st_size > 1000:
                    return True
                time.sleep(0.5)
        else:
            loop.run_until_complete(_edge_tts_save(text, voice, out_mp3))
            return True
    except Exception:
        pass
    
    txt_tmp = out_mp3.parent / "script_tts_tmp.txt"
    txt_tmp.write_text(text, encoding="utf-8")
    cmd = f'edge-tts --file "{txt_tmp}" --voice "{voice}" --write-media "{out_mp3}"'
    os.system(cmd)
    txt_tmp.unlink(missing_ok=True)
    return out_mp3.exists() and out_mp3.stat().st_size > 1000

# -----------------------------------------------------
# 3. THUẬT TOÁN VẼ PHỤ ĐỀ MÀU VÀNG NỔI BẬT CHÍNH GIỮA KHUNG HÌNH (OUTLINE ĐEN)
# -----------------------------------------------------
def draw_centered_yellow_subtitle(img_path, text, out_subtitle_img_path, font_size=55):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    
    font = None
    font_paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "C:\\Windows\\Fonts\\arialbd.ttf"
    ]
    for fp in font_paths:
        if os.path.exists(fp):
            try: font = ImageFont.truetype(fp, font_size); break
            except: pass
    if not font:
        font = ImageFont.load_default()
        
    words = text.split()
    lines = []
    curr = []
    for w in words:
        curr.append(w)
        if len(" ".join(curr)) > 24:
            lines.append(" ".join(curr[:-1]))
            curr = [w]
    if curr: lines.append(" ".join(curr))
    wrapped_text = "\n".join(lines)
    
    bbox = draw.multiline_textbbox((0, 0), wrapped_text, font=font, align="center")
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]
    
    x = (TARGET_W - tw) // 2
    y = (TARGET_H - th) // 2
    
    # Viền đen dày 5px nổi bật
    stroke_w = 5
    for dx in range(-stroke_w, stroke_w + 1):
        for dy in range(-stroke_w, stroke_w + 1):
            if dx != 0 or dy != 0:
                draw.multiline_text((x + dx, y + dy), wrapped_text, font=font, fill="black", align="center")
                
    # Chữ màu vàng rực rỡ chuẩn YouTube Shorts (#FFE600)
    draw.multiline_text((x, y), wrapped_text, font=font, fill="#FFE600", align="center")
    img.save(out_subtitle_img_path, "JPEG", quality=95)

# -----------------------------------------------------
# 4. TRÌNH RENDER VIDEO SHORT MP4 DỌC & TỰ ĐỘNG DỌN RÁC
# -----------------------------------------------------
def render_mp4_video(timeline, audio_path, out_mp4_path):
    print(f"🎥 Đang Render Video Short MP4 Dọc (1080x1920) với Phụ Đề Vàng Nổi Bật Chính Giữa Khung...")
    audio_clip = AudioFileClip(str(audio_path))
    total_duration = audio_clip.duration
    single_duration = total_duration / len(timeline)
    
    temp_dir = out_mp4_path.parent / f"_sub_frames_{int(time.time())}"
    temp_dir.mkdir(parents=True, exist_ok=True)
    
    image_clips = []
    for idx, item in enumerate(timeline):
        sub_frame_path = temp_dir / f"frame_{idx:03d}.jpg"
        draw_centered_yellow_subtitle(item['image_path'], item['text'], sub_frame_path)
        ic = ImageClip(str(sub_frame_path)).set_duration(single_duration)
        image_clips.append(ic)
        
    video = concatenate_videoclips(image_clips, method="compose")
    video = video.set_audio(audio_clip)
    
    video.write_videofile(
        str(out_mp4_path),
        fps=24,
        codec='libx264',
        audio_codec='aac',
        threads=4,
        preset='ultrafast',
        logger=None
    )
    audio_clip.close()
    video.close()
    shutil.rmtree(temp_dir, ignore_errors=True)
    print(f"   ✅ ĐÃ RENDER HOÀN TẤT FILE VIDEO SHORT MP4 CÓ PHỤ ĐỀ VÀNG: {out_mp4_path.name} ({out_mp4_path.stat().st_size // (1024*1024)} MB)")

def generate_video_short(anime_name, topic, api_key, voice="en-US-ChristopherNeural"):
    print(f"\n🎬 BẮT ĐẦU SẢN XUẤT VIDEO SHORT ANIME TIẾNG ANH CHO: {anime_name}")
    out_dir = BASE_LIBRARY_DIR / anime_name / "output_shorts"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    anime_dir = BASE_LIBRARY_DIR / anime_name
    conf_path = anime_dir / "characters_config.json"
    available_chars = list(json.loads(conf_path.read_text(encoding="utf-8")).keys()) if conf_path.exists() else []
    
    print("1/5. AI Gemini đang viết Kịch bản Tiếng Anh & Phân bổ 30 phân cảnh (2s/ảnh)...")
    script_data = generate_script_gemini(topic, anime_name, available_chars, api_key)
    if not script_data or 'scenes' not in script_data:
        print("❌ LỖI: Không thể tạo kịch bản từ Gemini API. Vui lòng kiểm tra API Key!"); return
        
    scenes = script_data['scenes']
    script_text = script_data.get('tts_script') or script_data.get('script', '')
    print(f"   ✅ Kịch bản Tiếng Anh AI hoàn tất: {len(script_text.split())} từ, {len(scenes)} phân cảnh!")
    
    print("2/5. Đang lựa chọn ngẫu nhiên 30 ảnh nhân vật KHÔNG TRÙNG LẶP cho từng 2 giây...")
    timeline = pick_unique_scene_images(scenes, anime_name)
    for item in timeline[:5]:
        print(f"   + Cảnh #{item['scene']:02d} (2s): [{item['character']}] ➡️ Ảnh: {item['image']}")
    print(f"   ... và {len(timeline)-5} phân cảnh tiếp theo độc bản không trùng!")
    
    print("3/5. Đang tạo giọng đọc Tiếng Anh Edge-TTS...")
    audio_temp_path = out_dir / "_temp_audio.mp3"
    ok = generate_tts_robust(script_text, voice, audio_temp_path)
    if not ok:
        print("❌ Không thể tạo file âm thanh TTS!"); return
    print(f"   ✅ Đã tạo giọng đọc Tiếng Anh thành công!")
    
    print("4/5. Đang tiến hành vẽ Phụ đề Vàng nổi bật chính giữa & Ghép Video Short MP4 Dọc...")
    timestamp = int(time.time())
    out_mp4_path = out_dir / f"{anime_name}_Short_{timestamp}.mp4"
    render_mp4_video(timeline, audio_temp_path, out_mp4_path)
    
    print("5/5. Đang tự động dọn dẹp các file rác (chỉ giữ duy nhất file MP4 hoàn chỉnh)...--------------")
    if audio_temp_path.exists(): audio_temp_path.unlink()
    for tmp_f in out_dir.glob("*_tmp*"):
        tmp_f.unlink(missing_ok=True)
        
    print(f"\n🎉 HOÀN THÀNH 100%! DƯỚI ĐÂY LÀ FILE VIDEO MP4 CÓ PHỤ ĐỀ VÀNG GIỮA MÀN HÌNH CHUẨN YOUTUBE SHORTS:")
    print(f"📹 Đường dẫn MP4 Google Drive: {out_mp4_path}")

In [ ]:
# @title 🎨 3. ANIME SHORT STUDIO WEB APP (PHỤ ĐỀ VÀNG VIỀN ĐEN NỔI BẬT GIỮA MÀN HÌNH CHUẨN YOUTUBE SHORTS)
import ipywidgets as widgets
from IPython.display import display, clear_output
import json, shutil
from pathlib import Path

BASE_LIBRARY_DIR = Path('/content/drive/MyDrive/anime_library')
DEFAULT_DATA = {
    "Tensei_Slime": {
        "Rimuru_Tempest": ["Rimuru Tempest"],
        "Veldora_Tempest": ["Veldora Tempest"],
        "Benimaru": ["Benimaru"],
        "Shuna": ["Shuna"],
        "Shion": ["Shion"],
        "Souei": ["Souei"],
        "Hakurou": ["Hakurou"],
        "Ranga": ["Ranga"],
        "Geld": ["Geld"],
        "Gabiru": ["Gabiru"],
        "Gobta": ["Gobta"],
        "Diablo": ["Diablo"],
        "Testarossa": ["Testarossa"],
        "Carrera": ["Carrera"],
        "Ultima": ["Ultima"],
        "Guy_Crimson": ["Guy Crimson"],
        "Rain": ["Rain"],
        "Misery": ["Misery"],
        "Milim_Nava": ["Milim Nava"],
        "Luminous_Valentine": ["Luminous Valentine"],
        "Ramiris": ["Ramiris"],
        "Leon_Cromwell": ["Leon Cromwell"],
        "Dagruel": ["Dagruel"],
        "Dino": ["Dino"],
        "Carrion": ["Carrion"],
        "Frey": ["Frey"],
        "Velgrynd": ["Velgrynd"],
        "Velzard": ["Velzard"],
        "Veldanava": ["Veldanava"],
        "Chloe_Aubert": ["Chloe Aubert"],
        "Hinata_Sakaguchi": ["Hinata Sakaguchi"],
        "Zegion": ["Zegion"],
        "Kumara": ["Kumara"],
        "Adalman": ["Adalman"],
        "Beretta": ["Beretta"],
        "Yuuki_Kagurazaka": ["Yuuki Kagurazaka"],
        "Feldway": ["Feldway"],
        "Rudra_Nam_Ul_Nasca": ["Rudra Nam Ul Nasca"]
    }
}
config_data = {}

def load_config():
    global config_data
    config_data = {}
    BASE_LIBRARY_DIR.mkdir(parents=True, exist_ok=True)
    
    legacy_config_path = BASE_LIBRARY_DIR / "anime_characters_config.json"
    if legacy_config_path.exists():
        try:
            with open(legacy_config_path, 'r', encoding='utf-8') as f:
                legacy_data = json.load(f)
                for anime_key, chars in legacy_data.items():
                    anime_dir = BASE_LIBRARY_DIR / anime_key
                    anime_dir.mkdir(parents=True, exist_ok=True)
                    char_conf_path = anime_dir / "characters_config.json"
                    if not char_conf_path.exists():
                        with open(char_conf_path, 'w', encoding='utf-8') as cf:
                            json.dump(chars, cf, indent=4, ensure_ascii=False)
        except Exception as e: print(f"Chuyển đổi config cũ: {e}")
    
    for item in BASE_LIBRARY_DIR.iterdir():
        if item.is_dir():
            anime_name = item.name
            char_conf = item / "characters_config.json"
            if char_conf.exists():
                try:
                    with open(char_conf, 'r', encoding='utf-8') as f:
                        config_data[anime_name] = json.load(f)
                except:
                    config_data[anime_name] = {}
    
    if "Tensei_Slime" not in config_data or not config_data["Tensei_Slime"]:
        config_data["Tensei_Slime"] = DEFAULT_DATA["Tensei_Slime"].copy()
        save_conf_for_anime("Tensei_Slime")
    else:
        updated = False
        for k, v in DEFAULT_DATA["Tensei_Slime"].items():
            if k not in config_data["Tensei_Slime"]:
                config_data["Tensei_Slime"][k] = v
                updated = True
        if updated:
            save_conf_for_anime("Tensei_Slime")

def save_conf_for_anime(anime_name):
    if anime_name in config_data:
        anime_dir = BASE_LIBRARY_DIR / anime_name
        anime_dir.mkdir(parents=True, exist_ok=True)
        conf_file = anime_dir / "characters_config.json"
        with open(conf_file, 'w', encoding='utf-8') as f:
            json.dump(config_data[anime_name], f, indent=4, ensure_ascii=False)

load_config()
out = widgets.Output()

# ==========================================
# TAB 1: QUẢN LÝ THƯ VIỆN & TẢI PINTEREST
# ==========================================
anime_dropdown = widgets.Dropdown(options=list(config_data.keys()), description='Chọn Anime:', layout=widgets.Layout(width='300px'))
new_anime_input = widgets.Text(placeholder='Anime Mới', layout=widgets.Layout(width='150px'))
add_anime_btn = widgets.Button(description='Thêm Anime', button_style='success')
del_anime_btn = widgets.Button(description='Xóa Cả Anime', button_style='danger')

new_char_input = widgets.Text(placeholder='Tên NV Mới', layout=widgets.Layout(width='150px'))
add_char_btn = widgets.Button(description='Thêm NV', button_style='info')

char_multiselect = widgets.SelectMultiple(
    options=[],
    description='Chọn Các NV:',
    layout=widgets.Layout(width='380px', height='160px')
)

del_char_btn = widgets.Button(description='💣 XÓA NV ĐƯỢC CHỌN & ẢNH', button_style='danger', layout=widgets.Layout(width='220px'))
target_count_slider = widgets.IntSlider(value=50, min=10, max=100, step=5, description='Số ảnh/NV:', layout=widgets.Layout(width='350px'))

fetch_selected_btn = widgets.Button(description='🚀 TẢI ẢNH CHO CÁC NV ĐƯỢC CHỌN (BẬT BẰNG GIỮ CTRL)', button_style='primary', layout=widgets.Layout(width='100%', height='45px'))
fetch_all_btn = widgets.Button(description='⚡ TẢI ẢNH CHO TẤT CẢ NV TRONG ANIME', button_style='success', layout=widgets.Layout(width='100%', height='45px'))

def update_ui(*args):
    sel_anime = anime_dropdown.value
    if sel_anime and sel_anime in config_data:
        char_list = list(config_data[sel_anime].keys())
        char_multiselect.options = char_list
        if char_list:
            char_multiselect.value = tuple(char_list)
        chars_str = ", ".join(char_list) if char_list else "(Trống)"
        with out:
            clear_output()
            conf_file = BASE_LIBRARY_DIR / sel_anime / "characters_config.json"
            print(f"📌 [Thư mục: /anime_library/{sel_anime}]")
            print(f"📄 Cấu hình lưu tại: {conf_file}")
            print(f"👉 Các nhân vật ({len(char_list)}): {chars_str}")

anime_dropdown.observe(update_ui, 'value')

def on_add_anime(b):
    nv = new_anime_input.value.strip().replace(" ", "_")
    if nv and nv not in config_data:
        config_data[nv] = {}
        save_conf_for_anime(nv)
        anime_dropdown.options = list(config_data.keys())
        anime_dropdown.value = nv
        new_anime_input.value = ''
        update_ui()

def on_del_anime(b):
    sel = anime_dropdown.value
    if sel in config_data:
        del config_data[sel]
        anime_dir = BASE_LIBRARY_DIR / sel
        if anime_dir.exists(): shutil.rmtree(anime_dir, ignore_errors=True)
        anime_dropdown.options = list(config_data.keys())
        if config_data: anime_dropdown.value = list(config_data.keys())[0]
        update_ui()
        with out: print(f"🗑️ Đã xóa Anime '{sel}' và toàn bộ thư mục Drive!")

def on_add_char(b):
    sel_anime = anime_dropdown.value
    cname = new_char_input.value.strip().replace(" ", "_")
    if sel_anime and cname:
        config_data[sel_anime][cname] = [cname.replace("_", " ")]
        save_conf_for_anime(sel_anime)
        new_char_input.value = ''
        update_ui()

def on_del_char(b):
    sel_anime = anime_dropdown.value
    selected_chars = list(char_multiselect.value)
    if sel_anime and selected_chars:
        for cname in selected_chars:
            if cname in config_data.get(sel_anime, {}):
                del config_data[sel_anime][cname]
                char_dir = BASE_LIBRARY_DIR / sel_anime / cname
                if char_dir.exists(): shutil.rmtree(char_dir, ignore_errors=True)
        save_conf_for_anime(sel_anime)
        update_ui()
        with out: print(f"💣 ĐÃ XÓA SẠCH {len(selected_chars)} nhân vật được chọn và thư mục ảnh Drive!")

def on_fetch_selected(b):
    with out:
        clear_output()
        sel_anime = anime_dropdown.value
        selected_chars = list(char_multiselect.value)
        target = target_count_slider.value
        if sel_anime and selected_chars:
            print(f"⏳ Đang cào riêng ảnh Pinterest cho {len(selected_chars)} NV được chọn trong '{sel_anime}'...")
            run_fetch(sel_anime, char_list=selected_chars, target_per_char=target)
        else:
            print("⚠️ Hãy giữ Ctrl và bấm chọn ít nhất 1 nhân vật trong danh sách trên!")

def on_fetch_all(b):
    with out:
        clear_output()
        sel_anime = anime_dropdown.value
        target = target_count_slider.value
        print(f"⏳ Đang cào ảnh Pinterest cho TẤT CẢ nhân vật trong '{sel_anime}' (Chỉ tiêu: {target} ảnh/NV)...")
        run_fetch(sel_anime, char_list=None, target_per_char=target)

add_anime_btn.on_click(on_add_anime); del_anime_btn.on_click(on_del_anime)
add_char_btn.on_click(on_add_char); del_char_btn.on_click(on_del_char)
fetch_selected_btn.on_click(on_fetch_selected); fetch_all_btn.on_click(on_fetch_all)

tab1_content = widgets.VBox([
    widgets.HTML("<h3>📁 CẤU HÌNH ANIME (LƯU RIÊNG THEO THƯ MỤC CỦA TỪNG ANIME)</h3>"),
    widgets.HBox([anime_dropdown, new_anime_input, add_anime_btn, del_anime_btn]),
    widgets.HBox([widgets.Label("Thêm NV Mới:"), new_char_input, add_char_btn]),
    widgets.HBox([char_multiselect, widgets.VBox([widgets.HTML("<i>💡 Giữ phím <b>Ctrl</b> hoặc <b>Shift</b> để chọn nhiều nhân vật cùng lúc!</i>"), del_char_btn])]),
    target_count_slider,
    fetch_selected_btn,
    fetch_all_btn
])

# ==========================================
# TAB 2: STUDIO TẠO VIDEO SHORT ANIME (BẮT BUỘC TIẾNG ANH)
# ==========================================
gemini_key_input = widgets.Text(description='Gemini API:', placeholder='Dán API Key Gemini vào đây', layout=widgets.Layout(width='450px'))
topic_input = widgets.Text(description='Chủ đề Short:', value='Secrets of Rimuru Tempest when evolving into a True Demon Lord', layout=widgets.Layout(width='550px'))
voice_dropdown = widgets.Dropdown(
    options=[
        ('English - Male Voice (Christopher)', 'en-US-ChristopherNeural'),
        ('English - Male Voice (Guy)', 'en-US-GuyNeural'),
        ('English - Female Voice (Jenny)', 'en-US-JennyNeural'),
        ('English - Female Voice (Aria)', 'en-US-AriaNeural')
    ],
    description='Giọng Đọc:', layout=widgets.Layout(width='400px')
)
create_short_btn = widgets.Button(description='🎬 1-CLICK TẠO VIDEO SHORT MP4 DỌC (PHỤ ĐỀ VÀNG NỔI BẮT GIỮ MÀN HÌNH)', button_style='success', layout=widgets.Layout(width='100%', height='50px'))

def on_create_short(b):
    with out:
        clear_output()
        key = gemini_key_input.value.strip()
        if not key:
            print("⚠️ VUI LÒNG NHẬP GEMINI API KEY VÀO Ô 'Gemini API' TRƯỚC KHI TẠO VIDEO!")
            return
        sel_anime = anime_dropdown.value
        topic = topic_input.value.strip()
        voice = voice_dropdown.value
        print(f"🚀 Đang tiến hành Render Video Short MP4 Tiếng Anh cho Anime '{sel_anime}'...")
        generate_video_short(sel_anime, topic, key, voice)

create_short_btn.on_click(on_create_short)

tab2_content = widgets.VBox([
    widgets.HTML("<h3>🎬 XƯỞNG RENDER VIDEO SHORT MP4 ANIME (PHỤ ĐỀ VÀNG GIỮA KHUNG HÌNH - YOUTUBE SHORTS)</h3>"),
    gemini_key_input,
    topic_input,
    voice_dropdown,
    create_short_btn
])

# ==========================================
# GỘP THÀNH GIAO DIỆN TAB WEB STUDIO DỄ DÙNG
# ==========================================
tabs = widgets.Tab(children=[tab1_content, tab2_content])
tabs.set_title(0, '📁 1. Cấu hình & Tải Ảnh Pinterest')
tabs.set_title(1, '🎬 2. Tạo Video Short MP4 (English)')

if config_data: update_ui()

ui = widgets.VBox([
    widgets.HTML("<h2 style='color:#1E88E5;'>🌟 ANIME SHORT STUDIO WEB APP — PINTEREST & FULL MP4 MAKER (ENGLISH)</h2>"),
    tabs,
    out
])
display(ui)